# Notebook: nb_silver_customers
# Purpose: Clean + conform bronze_customers into silver_customers
# Layer: Silver (clean / conform)
# Engine: python (single-node polars / delta-rs)
# Source: bronze_customers (read EXCLUSIVELY via read_bronze)
# Target: silver_customers (full-refresh overwrite)

In [ ]:
# --- Shared config + helpers (read_bronze, table_path, add_silver_metadata, validate_row_count) ---
# %run references a NOTEBOOK ITEM by its bare name (works in Python notebooks);
# never a repo path like utilities/nb_utils_config — deploys flatten the tree, so
# only the bare item name resolves. Keeps path / abfss literals inside utilities.
%run nb_utils_config

In [ ]:
# --- Imports ---
import polars as pl
from deltalake import write_deltalake

In [ ]:
# --- Configuration ---
TABLE_NAME = silver_table("customers")
BRONZE_SOURCE = "customers"

In [ ]:
# --- Read from bronze layer (the ONLY allowed read path) ---
df_raw = read_bronze(BRONZE_SOURCE)

print(f"Bronze rows: {df_raw.height:,}")
print(f"Bronze columns: {df_raw.columns}")

In [ ]:
# --- Rename + cast (polars; snake_case, proper types) ---
df_clean = df_raw.rename({
    "CustomerID": "customer_id",
    "FullName": "full_name",
    "SignupDate": "signup_date",
    "LifetimeValue": "lifetime_value",
    "StatusCode": "status_code",
}).with_columns(
    pl.col("customer_id").cast(pl.Int64),
    pl.col("signup_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("lifetime_value").cast(pl.Decimal(19, 4)),
)

In [ ]:
# --- Decode categorical + handle nulls (when/then/otherwise; fill_null) ---
df_clean = df_clean.with_columns(
    pl.when(pl.col("status_code") == "A").then(pl.lit("Active"))
      .when(pl.col("status_code") == "I").then(pl.lit("Inactive"))
      .otherwise(pl.lit("Unknown")).alias("status"),
).with_columns(
    pl.col("lifetime_value").fill_null(0),
    pl.col("full_name").fill_null("Unknown"),
)

In [ ]:
# --- Deduplicate: keep the latest row per customer_id ---
# Sort by the bronze load timestamp descending, then keep the first per key.
df_clean = df_clean.sort("_load_timestamp", descending=True).unique(
    subset=["customer_id"], keep="first", maintain_order=True
)

In [ ]:
# --- Filter invalid / test rows ---
df_clean = df_clean.filter(
    ~pl.col("full_name").str.contains(r"(?i)^test")
).filter(
    pl.col("lifetime_value") >= 0
)

In [ ]:
# --- Drop bronze metadata + add silver metadata ---
# add_silver_metadata() drops _load_timestamp/_source_file/_load_id if present
# and stamps _silver_processed_timestamp.
df_silver = add_silver_metadata(df_clean)

In [ ]:
# --- Write to silver Delta table (overwrite + schema overwrite; path via table_path()) ---
write_deltalake(
    table_path(TABLE_NAME),
    df_silver.to_arrow(),
    mode="overwrite",
    schema_mode="overwrite",
)

print(f"Written to: {TABLE_NAME}")

In [ ]:
# --- Validation ---
validate_row_count(TABLE_NAME, min_rows=1)
print("PASS: Silver transform complete")